# Racing drone: first camera films of AirSim and Project AirSim

This notebook makes the first camera films of two drone simulators: **AirSim** and **Project AirSim**. Both draw their pictures with Unreal Engine, which needs a graphics card. Colab's free T4 graphics card has one.

**How to use it:** choose *Runtime > Change runtime type > T4 GPU > Save*, then *Runtime > Run all*. Allow access to Google Drive when it asks. Keep this tab open until the last cell prints **Done**. The run takes about 30 to 45 minutes.

**What you get:** for each simulator, three films (the drone's own camera, a camera that follows the drone, and the drone's camera beside a map of the path), three sheets of 12 pictures from those films, a short summary in plain English, and all log files. They are saved on your Google Drive, in the folder `github/Comprehansive_cv/coursera_mirror/mirrors/Autonomous_Racing_Drone_Path/runs/2026-10-15_colab_unreal/<date_time>/`.

If one simulator fails, the other one still runs. Every failure is written into that simulator's `summary.md` together with the last 60 lines of its logs.

## Step 0. Settings and small helper functions (nothing to change here)

In [ ]:
import datetime
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import threading
import time
import traceback
from pathlib import Path

TEST = os.environ.get("UNREAL_FILMS_TEST") == "1"          # test switch: no GPU, no Drive, simulators start with -nullrhi
WORK = Path(os.environ.get("UNREAL_FILMS_WORK", "/content"))  # downloads and unpacked simulators live here, never on Drive
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d_%H%M")
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/github/Comprehansive_cv/coursera_mirror/mirrors/Autonomous_Racing_Drone_Path/runs/2026-10-15_colab_unreal")
LOCAL_RESULTS = Path(os.environ.get("UNREAL_FILMS_OUT", str(WORK / "results"))) / STAMP
DRIVE_RESULTS = None                                         # set in step 3
FPS = 15                                                     # pictures per simulated second in the films
RTF_PROJECT_AIRSIM = 1.0 if TEST else 0.1                    # Project AirSim clock speed (0.1 = ten times slower than real time)
MAX_TOTAL_MINUTES = 40                                       # no new simulator is started after this
T_START = time.time()
AIRSIM_URL = "https://github.com/microsoft/AirSim/releases/download/v1.8.1/Blocks.zip"
PASIM_URL = "https://github.com/iamaisim/ProjectAirSim/releases/download/v1.0.1/Blocks-Linux-1.0.1.zip"
IS_ROOT = hasattr(os, "geteuid") and os.geteuid() == 0
SIM_USER = "ue"                                              # Unreal Engine refuses to run as root, so the simulators run as this plain user
S = {"test": TEST, "stamp": STAMP, "vulkan": {}, "sims": {}}  # facts collected during the run


class StopRun(Exception):
    pass


def say(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def sh(cmd, log=None, timeout=None, env=None, cwd=None, quiet=True):
    """Run a shell command. Output goes to the log file (appended). Returns the exit code (124 on timeout)."""
    f = open(log, "ab") if log else subprocess.DEVNULL
    if log:
        f.write(f"\n$ {cmd}\n".encode())
        f.flush()
    try:
        r = subprocess.run(["bash", "-c", cmd], stdout=f, stderr=subprocess.STDOUT, timeout=timeout, env=env, cwd=cwd)
        return r.returncode
    except subprocess.TimeoutExpired:
        return 124
    finally:
        if log:
            f.close()


def out(cmd, timeout=120):
    """Run a shell command and return (exit code, text)."""
    try:
        r = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True, timeout=timeout)
        return r.returncode, (r.stdout + r.stderr)
    except subprocess.TimeoutExpired:
        return 124, "timeout"


def tail(path, n=60):
    try:
        return "".join(Path(path).read_text(errors="replace").splitlines(keepends=True)[-n:])
    except Exception as e:  # noqa: BLE001
        return f"(could not read {path}: {e!r})"


def apt_install(pkgs, log):
    """Install packages; if the group fails, one by one so a missing name does not block the rest. Returns the names that failed."""
    rc = sh("DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends " + " ".join(pkgs), log)
    if rc == 0:
        return []
    failed = []
    for p in pkgs:
        if sh(f"DEBIAN_FRONTEND=noninteractive apt-get install -y -qq --no-install-recommends {p}", log) != 0:
            failed.append(p)
    return failed


_synced = {}


def sync_to_drive():
    """Copy new or changed result files to Drive (so a crash still leaves the logs there)."""
    if DRIVE_RESULTS is None or not LOCAL_RESULTS.exists():
        return
    for p in LOCAL_RESULTS.rglob("*"):
        if not p.is_file():
            continue
        try:
            st = p.stat()
            key = (st.st_size, int(st.st_mtime))
            if _synced.get(p) == key:
                continue
            dest = DRIVE_RESULTS / p.relative_to(LOCAL_RESULTS)
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(p, dest)
            _synced[p] = key
        except Exception:  # noqa: BLE001
            pass


def _sync_loop():
    while True:
        time.sleep(20)
        try:
            sync_to_drive()
        except Exception:  # noqa: BLE001
            pass


def kill_group(p):
    if p is None or p.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(os.getpgid(p.pid), sig)
        except Exception:  # noqa: BLE001
            pass
        try:
            p.wait(timeout=8)
            return
        except Exception:  # noqa: BLE001
            pass


LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)
(WORK / "helpers").mkdir(parents=True, exist_ok=True)
say("test mode:", TEST, "| working folder:", WORK, "| results (local):", LOCAL_RESULTS)

## Step 1. Check that this computer has a graphics card

In [ ]:
try:
    if TEST:
        say("TEST MODE: the graphics card check is skipped.")
        S["gpu"] = {"name": "none (test mode)", "driver": None}
    else:
        rc, txt = out("nvidia-smi")
        print(txt)
        if rc != 0 or "NVIDIA" not in txt:
            raise StopRun("There is no graphics card. Choose Runtime > Change runtime type > T4 GPU, then Run all again.")
        _, q = out("nvidia-smi --query-gpu=name,driver_version --format=csv,noheader")
        name, driver = [x.strip() for x in q.strip().splitlines()[0].split(",")]
        S["gpu"] = {"name": name, "driver": driver}
        say(f"Graphics card found: {name}, driver {driver}.")
except StopRun as e:
    print("\n" + str(e))
    raise

## Step 2. Turn on the graphics card's Vulkan driver (and install a few helper libraries)
Colab ships the card's compute driver but not its drawing driver. The simulators need the drawing driver (Vulkan). We install the package `libnvidia-gl-<N>` whose number matches the running driver, then check with `vulkaninfo`.

In [ ]:
def step_vulkan():
    log = LOCAL_RESULTS / "setup_vulkan.log"
    info = {"nvidia_vulkan_device_found": False, "installed": []}
    sh("apt-get update -qq", log)
    libs = ["vulkan-tools", "libvulkan1", "xvfb", "xauth", "unzip", "curl", "git", "libgl1", "libglu1-mesa", "libsdl2-2.0-0", "libxi6",
            "libxrandr2", "libxcursor1", "libxinerama1", "libnss3", "libasound2t64", "libasound2", "libgl1-mesa-dri"]
    if shutil.which("ffmpeg") is None:
        libs.append("ffmpeg")
    info["library_install_failed"] = apt_install(libs, log)   # one of libasound2t64 / libasound2 is expected to fail
    if TEST:
        info["note"] = "TEST MODE: the graphics driver is not installed; the simulators start with -nullrhi"
        S["vulkan"] = info
        return info
    driver = S["gpu"]["driver"]
    major = int(driver.split(".")[0])
    _, names = out("apt-cache pkgnames libnvidia-gl-")
    avail = sorted({int(m.group(1)) for l in names.split() for m in [re.fullmatch(r"libnvidia-gl-(\d+)", l.strip())] if m})
    info["driver"], info["driver_major"], info["packages_available"] = driver, major, avail
    order = sorted(avail, key=lambda v: (abs(v - major), v))
    if major not in avail:
        say(f"apt has no libnvidia-gl-{major}; the nearest available are {order[:3]}")
    for cand in order[:3]:
        _, mad = out(f"apt-cache madison libnvidia-gl-{cand}")
        pins = [l.split("|")[1].strip() for l in mad.splitlines() if len(l.split("|")) > 2 and l.split("|")[1].strip().startswith(driver)]
        tries = ([f"libnvidia-gl-{cand}={pins[0]}"] if pins else []) + [f"libnvidia-gl-{cand}"]
        done = False
        for t in tries:
            if sh(f"DEBIAN_FRONTEND=noninteractive apt-get install -y -qq {t}", log) == 0:
                info["installed"].append(t)
                done = True
                break
        if done:
            break
    _, dpk = out("dpkg -l 'libnvidia-gl-*' | grep '^ii'")
    info["dpkg"] = dpk.strip()
    say("installed:", info["installed"] or "NOTHING", "|", dpk.strip() or "no libnvidia-gl package")

    def check():
        _, vk = out("vulkaninfo --summary", timeout=120)
        (LOCAL_RESULTS / "vulkaninfo_summary.txt").write_text(vk)
        return bool(re.search(r"driverName\s*=\s*NVIDIA", vk) or re.search(r"deviceName\s*=.*(NVIDIA|Tesla)", vk)), vk

    found, vk = check()
    icd = next((p for p in ("/usr/share/vulkan/icd.d/nvidia_icd.json", "/etc/vulkan/icd.d/nvidia_icd.json") if Path(p).is_file()), None)
    if not found and icd is None:
        libs_found = out("ldconfig -p | grep -i libGLX_nvidia")[1].strip()
        if libs_found:
            Path("/usr/share/vulkan/icd.d").mkdir(parents=True, exist_ok=True)
            Path("/usr/share/vulkan/icd.d/nvidia_icd.json").write_text(
                json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libGLX_nvidia.so.0", "api_version": "1.3.0"}}))
            icd = "/usr/share/vulkan/icd.d/nvidia_icd.json"
            info["wrote_icd_file_by_hand"] = True
            found, vk = check()
    info["nvidia_icd_file"] = icd
    info["nvidia_vulkan_device_found"] = found
    info["vulkaninfo_devices"] = [l.strip() for l in vk.splitlines() if "deviceName" in l or "driverName" in l]
    say("Vulkan lists the NVIDIA card:", found, info["vulkaninfo_devices"])
    S["vulkan"] = info
    return info


try:
    step_vulkan()
except Exception:  # noqa: BLE001
    S["vulkan"]["error"] = traceback.format_exc()
    say("Step 2 had a problem; the run goes on, but the simulators may fail:\n" + S["vulkan"]["error"][-800:])
(LOCAL_RESULTS / "vulkan.json").write_text(json.dumps(S["vulkan"], indent=2, default=str))

## Step 3. Connect Google Drive and make the results folder
Colab will ask you to allow access to your Google Drive. Please click **Connect to Google Drive** and choose your account.

In [ ]:
if TEST:
    say("TEST MODE: Google Drive is skipped; results stay in", LOCAL_RESULTS)
else:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
    DRIVE_RESULTS = DRIVE_RUN_ROOT / STAMP
    for sub in ("airsim", "project_airsim"):
        (DRIVE_RESULTS / sub).mkdir(parents=True, exist_ok=True)
    say("Results folder on Drive:", DRIVE_RESULTS)
    threading.Thread(target=_sync_loop, daemon=True).start()
for sub in ("airsim", "project_airsim"):
    (LOCAL_RESULTS / sub).mkdir(parents=True, exist_ok=True)
S["drive_results"] = str(DRIVE_RESULTS) if DRIVE_RESULTS else None
sync_to_drive()

## Step 3b. Write the helper programs (the flight route, the film maker, the two simulator clients)

In [ ]:
HELPERS = {
    "route.py": r'''"""The flight route, the same for both simulators. Pure Python, no packages.

Phases (seconds since the start of the recording):
  0 to CLIMB_S             climb straight up to HEIGHT_M above the start (nose along the start heading)
  CLIMB_S to CLIMB_S+LOOP_S   one figure eight (a Gerono lemniscate), 12 m north-south and 6 m east-west, nose along the direction of flight
  then until TOTAL_S       hover at the end of the loop

Frames: north, east, down (NED), the same as both simulators. yaw = 0 points north, positive turns toward east.
"""

import math

HEIGHT_M = 3.0
CLIMB_S = 6.0
LOOP_S = 24.0
TOTAL_S = 36.0
A_M = 6.0   # figure eight: north = A sin u        (12 m long)
B_M = 6.0   # figure eight: east  = B sin u cos u  (3 m to each side)
KP = 1.0
V_MAX = 3.0
YAW_RATE_MAX = math.radians(90.0)


def _ease(tau: float):
    """Smooth 0..1 progress with zero speed at both ends: s = tau - sin(2 pi tau) / (2 pi). Returns (s, ds/dtau)."""
    tau = min(max(tau, 0.0), 1.0)
    return tau - math.sin(2 * math.pi * tau) / (2 * math.pi), 1.0 - math.cos(2 * math.pi * tau)


def loop_point(u: float):
    """Position and derivative d/du of the figure eight at angle u, relative to the loop centre (north, east)."""
    n = A_M * math.sin(u)
    e = B_M * math.sin(u) * math.cos(u)
    dn = A_M * math.cos(u)
    de = B_M * math.cos(2 * u)
    return (n, e), (dn, de)


def plan_xy(start_n: float, start_e: float, points: int = 200):
    """The planned horizontal path for the map (north, east)."""
    pts = [[start_n, start_e]]
    for k in range(points + 1):
        (n, e), _ = loop_point(2 * math.pi * k / points)
        pts.append([start_n + n, start_e + e])
    return pts


def target(t: float, start_n: float, start_e: float, start_d: float):
    """Where the drone should be at time t: (north, east, down, v_north, v_east)."""
    z = start_d - HEIGHT_M
    if t < CLIMB_S:
        return start_n, start_e, z, 0.0, 0.0
    tau = (t - CLIMB_S) / LOOP_S
    s, ds = _ease(tau)
    u = 2 * math.pi * s
    du_dt = 2 * math.pi * ds / LOOP_S
    (n, e), (dn, de) = loop_point(u)
    return start_n + n, start_e + e, z, dn * du_dt, de * du_dt


class Controller:
    """Velocity command = feed-forward + KP * position error, clipped to V_MAX; yaw follows the velocity, rate limited."""

    def __init__(self, start_n: float, start_e: float, start_d: float, start_yaw: float):
        self.s = (start_n, start_e, start_d)
        self.yaw = start_yaw
        self.last_t = None

    def command(self, t: float, n: float, e: float):
        """Returns (v_north, v_east, z_down_target, yaw_rad) for time t and the measured north/east position."""
        tn, te, tz, fvn, fve = target(t, *self.s)
        vn = fvn + KP * (tn - n)
        ve = fve + KP * (te - e)
        sp = math.hypot(vn, ve)
        if sp > V_MAX:
            vn, ve = vn * V_MAX / sp, ve * V_MAX / sp
        if t >= CLIMB_S and math.hypot(fvn, fve) > 0.4:
            want = math.atan2(fve, fvn)  # direction of the PLANNED motion, so the nose does not wobble with the error
            dt = 0.0 if self.last_t is None else max(t - self.last_t, 0.0)
            diff = (want - self.yaw + math.pi) % (2 * math.pi) - math.pi
            step = max(-YAW_RATE_MAX * dt, min(YAW_RATE_MAX * dt, diff))
            self.yaw += step
        self.last_t = t
        return vn, ve, tz, self.yaw


def quat_wxyz_to_yaw(w: float, x: float, y: float, z: float) -> float:
    return math.atan2(2 * (w * z + x * y), 1 - 2 * (y * y + z * z))''',
    "unreal_films.py": r'''"""Films, contact sheets and the map for the two Unreal simulators (AirSim and Project AirSim).

Input: a folder with the pictures a flight program saved and a track.json (see below).
Output: fpv.mp4, chase.mp4, fpv_and_map.mp4, fpv_contact.png, chase_contact.png, fpv_and_map_contact.png.

track.json = {"fps": 15, "plan_xy": [[north, east], ...], "start_xy": [n, e],
              "samples": [{"t": sim seconds, "n": north m, "e": east m, "d": down m, "yaw": rad (0 = north, + toward east),
                           "fpv": "file name or null", "chase": "file name or null"}, ...]}

The film code is the same as in the PX4 + Gazebo and Webots films (branch px4-gazebo-base, tools/sync_film.py): H.264 yuv420p
through ffmpeg, a contact sheet of 12 pictures (4 by 3, first and last included) stamped with the simulated time, and a map from
above beside the camera picture. The map here has no gates: it shows the planned path (thin), the path flown so far (bold),
the drone (dot) and the direction of the nose (arrow). East is to the right, north is up.

The film is made at a steady rate: picture k of the film shows the sample whose simulated time is closest to k / fps. If the
simulator delivered fewer pictures than that, a picture is shown twice; the summary reports how many different pictures there were.

  python unreal_films.py --selftest OUT_DIR TRACK_JSON    (makes films from drawn stand-in pictures, never from a simulator)
"""

from __future__ import annotations

import json
import math
import subprocess
import sys
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw, ImageFont

BACKGROUND = (247, 247, 244)
GRID = (225, 225, 220)
GRID_TEXT = (150, 150, 145)
PLAN_COLOR = (110, 110, 215)
FLOWN_COLOR = (214, 39, 40)
DOT_COLOR = (0, 0, 0)
ARROW_COLOR = (0, 160, 70)
TEXT_COLOR = (20, 20, 20)
START_FILL = (255, 235, 59)
START_LINE = (120, 110, 0)
FOOTER_PX = 36
ARROW_LENGTH_PX = 34


def font(size: int):
    try:
        return ImageFont.load_default(size=size)
    except TypeError:  # Pillow older than 10.1
        return ImageFont.load_default()


class FilmWriter:
    """Raw RGB pictures in on stdin, H.264 yuv420p mp4 out."""

    def __init__(self, path: Path, width: int, height: int, fps: int, log_path: Path | None = None):
        if width % 2 or height % 2:
            raise ValueError(f"yuv420p needs even picture sizes, got {width}x{height}")
        self.path, self.width, self.height = Path(path), int(width), int(height)
        self.count = 0
        self._log = open(log_path, "ab") if log_path else subprocess.DEVNULL
        cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{width}x{height}",
               "-r", str(fps), "-i", "-", "-c:v", "libx264", "-preset", "veryfast", "-crf", "23",
               "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(self.path)]
        self._proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=self._log)

    def write(self, img: np.ndarray) -> None:
        if img.shape != (self.height, self.width, 3) or img.dtype != np.uint8:
            raise ValueError(f"unexpected picture {img.shape} {img.dtype}, wanted ({self.height}, {self.width}, 3) uint8")
        self._proc.stdin.write(np.ascontiguousarray(img).tobytes())
        self.count += 1

    def close(self) -> None:
        try:
            self._proc.stdin.close()
        except Exception:
            pass
        code = self._proc.wait(timeout=600)
        if hasattr(self._log, "close"):
            self._log.close()
        if code != 0 or not self.path.is_file() or self.path.stat().st_size == 0:
            raise RuntimeError(f"ffmpeg failed with return code {code} for {self.path}")


def _read_exact(stream, n: int) -> bytes:
    chunks, got = [], 0
    while got < n:
        chunk = stream.read(n - got)
        if not chunk:
            break
        chunks.append(chunk)
        got += len(chunk)
    return b"".join(chunks)


def read_frames_at(film: Path, width: int, height: int, wanted: list[int] | None = None) -> tuple[dict, int]:
    """Decode a film: ({index: picture} for the wanted indices, number of pictures ffmpeg decodes)."""
    wanted_set = set(wanted or [])
    proc = subprocess.Popen(["ffmpeg", "-loglevel", "error", "-i", str(film), "-f", "rawvideo", "-pix_fmt", "rgb24", "-"],
                            stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    size = width * height * 3
    found, index = {}, 0
    try:
        while True:
            raw = _read_exact(proc.stdout, size)
            if len(raw) < size:
                break
            if index in wanted_set:
                found[index] = np.frombuffer(raw, dtype=np.uint8).reshape(height, width, 3).copy()
            index += 1
    finally:
        proc.stdout.close()
        proc.wait(timeout=120)
    return found, index


def evenly_spaced_indices(count: int, n: int = 12) -> list[int]:
    if count <= 0:
        return []
    if count <= n:
        return list(range(count))
    return [int(round(k * (count - 1) / (n - 1))) for k in range(n)]


def contact_sheet(images: list[np.ndarray], labels: list[str], cols: int = 4, max_width: int = 1920) -> Image.Image:
    h, w = images[0].shape[:2]
    tile_w = max_width // cols
    tile_h = int(round(h * tile_w / w))
    tile_w -= tile_w % 2
    rows = (len(images) + cols - 1) // cols
    sheet = Image.new("RGB", (cols * tile_w, rows * tile_h), (30, 30, 30))
    f = font(max(12, tile_h // 14))
    for k, (img, label) in enumerate(zip(images, labels)):
        tile = Image.fromarray(img).resize((tile_w, tile_h), Image.BILINEAR)
        d = ImageDraw.Draw(tile)
        box = d.textbbox((6, 4), label, font=f)
        d.rectangle([box[0] - 4, box[1] - 3, box[2] + 4, box[3] + 3], fill=(0, 0, 0))
        d.text((6, 4), label, fill=(255, 255, 255), font=f)
        sheet.paste(tile, ((k % cols) * tile_w, (k // cols) * tile_h))
    return sheet


def film_contact_sheet(film: Path, width: int, height: int, times, out_png: Path, n: int = 12) -> dict:
    _, count = read_frames_at(film, width, height, None)
    idx = evenly_spaced_indices(count, n)
    frames, _ = read_frames_at(film, width, height, idx)
    images = [frames[i] for i in idx]
    labels = [f"{float(times[i]):.2f} s" if i < len(times) else f"picture {i}" for i in idx]
    sheet = contact_sheet(images, labels, cols=4)
    sheet.save(out_png)
    return {"pictures_decoded": count, "indices": idx, "file": Path(out_png).name, "sheet_size": list(sheet.size)}


class MapPanel:
    """Map from above, a square of `size` pixels (the lowest FOOTER_PX pixels hold one line of text). East right, north up."""

    def __init__(self, plan_ne, start_ne, all_ne, size: int = 480, margin_m: float = 2.0):
        self.size, self.footer = size, FOOTER_PX
        pts = np.asarray(list(plan_ne) + list(all_ne) + [start_ne], dtype=float).reshape(-1, 2)
        lo, hi = pts.min(axis=0) - margin_m, pts.max(axis=0) + margin_m
        area = size - self.footer
        self.scale = min(size / max(hi[1] - lo[1], 1.0), area / max(hi[0] - lo[0], 1.0))
        cx, cn = (lo[1] + hi[1]) / 2, (lo[0] + hi[0]) / 2
        self.origin = (size / 2 - self.scale * cx, area / 2 + self.scale * cn)  # pixel of east = 0, north = 0
        self.font_small, self.font_text = font(12), font(15)
        self.plan_ne, self.start_ne = np.asarray(plan_ne, dtype=float).reshape(-1, 2), np.asarray(start_ne, dtype=float)
        self.background_image = self._draw_static()

    def px(self, ne) -> tuple[float, float]:
        return (self.origin[0] + self.scale * float(ne[1]), self.origin[1] - self.scale * float(ne[0]))

    def _draw_static(self) -> Image.Image:
        img = Image.new("RGB", (self.size, self.size), BACKGROUND)
        d = ImageDraw.Draw(img)
        area = self.size - self.footer
        step = 5 if self.size / self.scale > 25 else 2
        e_lo, e_hi = math.floor((0 - self.origin[0]) / self.scale), math.ceil((self.size - self.origin[0]) / self.scale)
        for e in range(e_lo - e_lo % step, e_hi + 1, step):
            u = self.origin[0] + self.scale * e
            if 0 <= u < self.size:
                d.line([(u, 0), (u, area)], fill=GRID, width=1)
                d.text((u + 2, area - 14), f"{e}", fill=GRID_TEXT, font=self.font_small)
        n_lo, n_hi = math.floor((self.origin[1] - area) / self.scale), math.ceil(self.origin[1] / self.scale)
        for n in range(n_lo - n_lo % step, n_hi + 1, step):
            v = self.origin[1] - self.scale * n
            if 0 <= v < area:
                d.line([(0, v), (self.size, v)], fill=GRID, width=1)
                d.text((2, v - 14), f"{n}", fill=GRID_TEXT, font=self.font_small)
        d.text((self.size - 52, 4), "east >", fill=GRID_TEXT, font=self.font_small)
        d.text((4, 4), "north ^", fill=GRID_TEXT, font=self.font_small)
        if len(self.plan_ne) > 1:
            d.line([self.px(p) for p in self.plan_ne], fill=PLAN_COLOR, width=1, joint="curve")
        u, v = self.px(self.start_ne)
        r = 7
        d.ellipse([u - r, v - r, u + r, v + r], fill=START_FILL, outline=START_LINE, width=2)
        d.text((u - 12, v + r + 1), "start", fill=TEXT_COLOR, font=self.font_small)
        d.rectangle([0, area, self.size, self.size], fill=(255, 255, 255))
        d.line([(0, area), (self.size, area)], fill=GRID, width=1)
        return img

    @staticmethod
    def _arrow(d, p0, p1, color, width: int, head: int) -> None:
        d.line([p0, p1], fill=color, width=width)
        ang = math.atan2(p1[1] - p0[1], p1[0] - p0[0])
        for s in (-1, 1):
            a = ang + math.pi - s * math.radians(28)
            d.line([p1, (p1[0] + head * math.cos(a), p1[1] + head * math.sin(a))], fill=color, width=width)

    def render(self, flown_ne, yaw: float, time_s: float, speed_mps: float, alt_m: float) -> np.ndarray:
        img = self.background_image.copy()
        d = ImageDraw.Draw(img)
        pts = [self.px(p) for p in np.asarray(flown_ne, dtype=float).reshape(-1, 2)]
        if len(pts) > 1:
            d.line(pts, fill=FLOWN_COLOR, width=4, joint="curve")
        dot = pts[-1]
        tip = (dot[0] + ARROW_LENGTH_PX * math.sin(yaw), dot[1] - ARROW_LENGTH_PX * math.cos(yaw))
        self._arrow(d, dot, tip, ARROW_COLOR, 3, 11)
        d.ellipse([dot[0] - 6, dot[1] - 6, dot[0] + 6, dot[1] + 6], fill=DOT_COLOR)
        d.text((8, self.size - self.footer + 9), f"t {time_s:5.2f} s   speed {speed_mps:4.2f} m/s   height {alt_m:4.1f} m",
               fill=TEXT_COLOR, font=self.font_text)
        return np.asarray(img)


def _load(path: Path):
    if path is None:
        return None
    try:
        return np.asarray(Image.open(path).convert("RGB"))
    except Exception:
        return None


def make_films(out_dir: Path, frames_dir: Path, track: dict, log_path: Path | None = None) -> dict:
    """Write the three films and three contact sheets. Returns a dict of facts (never claims a film that was not written)."""
    out_dir, frames_dir = Path(out_dir), Path(frames_dir)
    fps = int(track.get("fps", 15))
    samples = [s for s in track["samples"] if s.get("t") is not None]
    info: dict = {"fps": fps, "samples": len(samples), "films": {}, "problems": []}
    if not samples:
        info["problems"].append("the flight program saved no samples")
        return info
    samples.sort(key=lambda s: s["t"])
    ts = np.array([s["t"] for s in samples])
    t_end = float(ts[-1])
    n_film = int(math.floor(t_end * fps)) + 1
    grid = np.arange(n_film) / fps
    nearest = np.abs(ts[None, :] - grid[:, None]).argmin(axis=1)
    info["film_seconds"] = n_film / fps
    all_ne = [[s["n"], s["e"]] for s in samples]
    # which cameras have pictures
    for cam in ("fpv", "chase"):
        have = [bool(s.get(cam)) and (frames_dir / s[cam]).is_file() for s in samples]
        info[f"{cam}_pictures_saved"] = int(sum(have))
        used = {int(i) for i in nearest if have[int(i)]}
        info[f"{cam}_different_pictures_in_film"] = len(used)
        info[f"{cam}_pictures_per_sim_second"] = (sum(have) / t_end) if t_end > 0 else None

    # speed from neighbouring samples
    pos = np.array([[s["n"], s["e"], s["d"]] for s in samples])
    speed = np.zeros(len(samples))
    for i in range(1, len(samples)):
        dt = max(ts[i] - ts[i - 1], 1e-6)
        speed[i] = float(np.linalg.norm(pos[i] - pos[i - 1]) / dt)

    def picture_for(idx: int, cam: str):
        """Picture of sample idx, or the nearest sample that has one (searching outward, at most 1 s away)."""
        for off in range(0, max(2, fps)):
            for j in (idx - off, idx + off):
                if 0 <= j < len(samples) and samples[j].get(cam):
                    img = _load(frames_dir / samples[j][cam])
                    if img is not None:
                        return img
        return None

    cache: dict = {}

    def cached(idx, cam):
        key = (idx, cam)
        if key not in cache:
            if len(cache) > 400:
                cache.clear()
            cache[key] = picture_for(idx, cam)
        return cache[key]

    sizes = {}
    for cam in ("fpv", "chase"):
        first = next((cached(int(i), cam) for i in nearest if cached(int(i), cam) is not None), None)
        if first is not None:
            h, w = first.shape[:2]
            sizes[cam] = (w - w % 2, h - h % 2)
    writers: dict = {}
    mp = None
    try:
        for cam, (w, h) in sizes.items():
            writers[cam] = FilmWriter(out_dir / f"{cam}.mp4", w, h, fps, log_path)
        if "fpv" in sizes:
            w, h = sizes["fpv"]
            mp = MapPanel(track.get("plan_xy", []), track.get("start_xy", all_ne[0]), all_ne, size=h)
            writers["fpv_and_map"] = FilmWriter(out_dir / "fpv_and_map.mp4", w + h, h, fps, log_path)
        for k in range(n_film):
            idx = int(nearest[k])
            for cam in ("fpv", "chase"):
                if cam in writers:
                    img = cached(idx, cam)
                    w, h = sizes[cam]
                    if img is None:
                        img = np.zeros((h, w, 3), np.uint8)
                    writers[cam].write(np.ascontiguousarray(img[:h, :w]))
            if "fpv_and_map" in writers:
                w, h = sizes["fpv"]
                img = cached(idx, "fpv")
                img = np.zeros((h, w, 3), np.uint8) if img is None else np.ascontiguousarray(img[:h, :w])
                upto = idx + 1
                mapimg = mp.render(pos[:upto, :2], samples[idx]["yaw"], float(ts[idx]), float(speed[idx]), float(-pos[idx, 2]))
                writers["fpv_and_map"].write(np.concatenate([img, mapimg], axis=1))
        for name, wr in writers.items():
            wr.close()
    except Exception as e:  # noqa: BLE001
        info["problems"].append(f"film writing failed: {e!r}")
        for wr in writers.values():
            try:
                wr._proc.kill()
            except Exception:
                pass
        writers = {}
    for name, wr in writers.items():
        w, h = (wr.width, wr.height)
        try:
            res = film_contact_sheet(out_dir / f"{name}.mp4", w, h, grid, out_dir / f"{name}_contact.png")
            res["pictures_written"] = wr.count
            res["bytes"] = (out_dir / f"{name}.mp4").stat().st_size
            info["films"][f"{name}.mp4"] = res
        except Exception as e:  # noqa: BLE001
            info["problems"].append(f"contact sheet of {name} failed: {e!r}")
    return info


def film_quality(out_dir: Path, info: dict) -> dict:
    """Pixel standard deviation of 12 pictures of each film and the mean difference between neighbours (the pass test of the other films)."""
    res = {}
    for name, f in info.get("films", {}).items():
        try:
            probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=width,height",
                                    "-of", "csv=p=0", str(Path(out_dir) / name)], capture_output=True, text=True)
            fw, fh = (int(x) for x in probe.stdout.strip().split(","))
            idx = evenly_spaced_indices(f["pictures_decoded"], 12)
            frames, _ = read_frames_at(Path(out_dir) / name, fw, fh, idx)
            arrs = [frames[i].astype(np.float64) for i in idx]
            res[name] = {"stds": [round(float(a.std()), 1) for a in arrs],
                         "neighbour_mean_abs_diff": [round(float(np.abs(a - b).mean()), 1) for a, b in zip(arrs[:-1], arrs[1:])]}
        except Exception as e:  # noqa: BLE001
            res[name] = {"error": repr(e)}
    return res


def selftest(out_dir: Path, track_json: Path) -> int:
    """Make films from drawn stand-in pictures (a coloured ground, a horizon that tilts with the yaw, the time in big letters)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    frames = out_dir / "standin_frames"
    frames.mkdir(exist_ok=True)
    track = json.loads(Path(track_json).read_text())
    fps = int(track.get("fps", 15))
    samples = track["samples"]
    # use at most 150 samples spread over the flight so the self-test is quick
    keep = sorted(set(int(round(k)) for k in np.linspace(0, len(samples) - 1, min(len(samples), 150))))
    samples = [samples[i] for i in keep]
    f = font(40)
    for i, s in enumerate(samples):
        for cam, size in (("fpv", (640, 480)), ("chase", (640, 480))):
            img = Image.new("RGB", size, (120, 170, 230) if cam == "fpv" else (150, 150, 150))
            d = ImageDraw.Draw(img)
            horizon = int(size[1] * 0.5 + 60 * math.sin(s["yaw"]))
            d.rectangle([0, horizon, size[0], size[1]], fill=(70, 130, 60) if cam == "fpv" else (90, 90, 90))
            for g in range(0, size[0], 80):
                d.line([(g + int(40 * math.cos(s["yaw"])), horizon), (g, size[1])], fill=(40, 90, 40), width=2)
            d.text((20, 20), f"STAND-IN {cam} t={s['t']:.2f}", fill=(255, 255, 255), font=f)
            name = f"{cam}_{i:05d}.jpg"
            img.save(frames / name, quality=90)
            s[cam] = name
    track["samples"] = samples
    info = make_films(out_dir, frames, track, out_dir / "ffmpeg.log")
    q = film_quality(out_dir, info)
    (out_dir / "selftest.json").write_text(json.dumps({"info": info, "quality": q}, indent=2))
    print(json.dumps({"info": info, "quality": q}, indent=2))
    ok = (not info["problems"] and {"fpv.mp4", "chase.mp4", "fpv_and_map.mp4"} <= set(info["films"])
          and all(v.get("pictures_decoded", 0) >= 50 for v in info["films"].values())
          and all(all(x > 5 for x in v.get("stds", [0])) for v in q.values()))
    print("SELFTEST", "PASS" if ok else "FAIL")
    return 0 if ok else 1


if __name__ == "__main__":
    if len(sys.argv) == 4 and sys.argv[1] == "--selftest":
        sys.exit(selftest(Path(sys.argv[2]), Path(sys.argv[3])))
    print(__doc__)''',
    "airsim_flight.py": r'''"""AirSim flight program (runs in the Python 3.9 environment that has the `airsim` package).

Connects to the running AirSim, checks that a picture arrives, flies the route of route.py while the simulation is PAUSED between
steps (pause, continueForTime(1/fps), read the state, take both pictures), and saves pictures and track.json.

Exit codes: 0 = flight done, 3 = no picture within the wait (the caller may start the simulator another way), 4 = no connection,
            1 = error during the flight (what was saved so far is kept).
Cameras: "front" = on the drone, looks along the nose (name in settings.json); "Chase" = external camera that is moved each step.
"""

import argparse
import json
import math
import sys
import time
import traceback
from pathlib import Path

import numpy as np
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parent))
import route  # noqa: E402

CHASE_BACK_M = 6.0
CHASE_UP_M = 2.5


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def to_rgb(resp):
    if resp.width == 0 or resp.height == 0 or len(resp.image_data_uint8) == 0:
        return None
    a = np.frombuffer(resp.image_data_uint8, np.uint8)
    ch = a.size // (resp.width * resp.height)
    a = a.reshape(resp.height, resp.width, ch)[:, :, :3][:, :, ::-1]  # AirSim delivers BGR(A); this is RGB
    return np.ascontiguousarray(a)


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    ap.add_argument("--fps", type=int, default=15)
    ap.add_argument("--connect-wait", type=float, default=240.0)
    ap.add_argument("--picture-wait", type=float, default=90.0)
    ap.add_argument("--test", action="store_true", help="no pictures expected: fly the route anyway")
    a = ap.parse_args()
    out = Path(a.out)
    frames = out / "frames"
    frames.mkdir(parents=True, exist_ok=True)
    res = {"stage": "start", "test": a.test, "traceback": None}

    def save_res():
        (out / "flight.json").write_text(json.dumps(res, indent=2, default=str))

    import airsim

    client = airsim.MultirotorClient(timeout_value=120)
    t0 = time.time()
    while True:
        try:
            client.confirmConnection()
            break
        except Exception as e:  # noqa: BLE001
            if time.time() - t0 > a.connect_wait:
                res["stage"] = "no_connection"
                res["error"] = repr(e)
                save_res()
                log("no connection:", e)
                return 4
            time.sleep(3)
    res["connected_after_s"] = round(time.time() - t0, 1)
    res["client_version"] = getattr(airsim, "__version__", None)
    log("connected after", res["connected_after_s"], "s")

    def grab(which):
        if which == "fpv":
            r = client.simGetImages([airsim.ImageRequest("front", airsim.ImageType.Scene, False, False)])
        else:
            r = client.simGetImages([airsim.ImageRequest("Chase", airsim.ImageType.Scene, False, False)], external=True)
        return to_rgb(r[0]) if r else None

    # ---- picture probe ------------------------------------------------------------
    res["stage"] = "probe"
    client.enableApiControl(True)
    client.armDisarm(True)
    got = {"fpv": None, "chase": None}
    errors = []
    t1 = time.time()
    while time.time() - t1 < a.picture_wait and (got["fpv"] is None or got["chase"] is None):
        for cam in ("fpv", "chase"):
            if got[cam] is None:
                try:
                    got[cam] = grab(cam)
                except Exception as e:  # noqa: BLE001
                    errors.append(f"{cam}: {e!r}"[:300])
        if got["fpv"] is None or got["chase"] is None:
            time.sleep(2)
    res["probe"] = {"fpv": got["fpv"] is not None, "chase": got["chase"] is not None, "seconds": round(time.time() - t1, 1),
                    "last_errors": errors[-3:]}
    log("probe:", res["probe"])
    if got["fpv"] is None and got["chase"] is None and not a.test:
        res["stage"] = "no_picture"
        save_res()
        return 3
    if got["fpv"] is not None:
        arr = got["fpv"].astype(float)
        top = arr[: arr.shape[0] // 5]
        res["probe"]["fpv_top_rows_mean_rgb"] = [round(float(x), 1) for x in top.reshape(-1, 3).mean(axis=0)]
        res["probe"]["fpv_std"] = round(float(arr.std()), 1)

    skip_grabs = a.test and got["fpv"] is None and got["chase"] is None
    res["skip_grabs"] = skip_grabs
    # ---- the flight ---------------------------------------------------------------
    res["stage"] = "flight"
    save_res()
    dt = 1.0 / a.fps
    samples, rc = [], 0
    try:
        st0 = client.getMultirotorState()
        p0 = st0.kinematics_estimated.position
        sn, se, sd = p0.x_val, p0.y_val, p0.z_val
        q0 = st0.kinematics_estimated.orientation
        yaw0 = route.quat_wxyz_to_yaw(q0.w_val, q0.x_val, q0.y_val, q0.z_val)
        ctl = route.Controller(sn, se, sd, yaw0)
        res["start"] = {"n": sn, "e": se, "d": sd, "yaw": yaw0}
        ts0 = st0.timestamp
        client.simPause(True)
        nsteps = int(round(route.TOTAL_S * a.fps))
        wall0 = time.perf_counter()
        pics = {"fpv": 0, "chase": 0}
        collisions = 0
        cam_fail = 0
        n, e = sn, se
        for step in range(nsteps + 1):
            t = step * dt
            vn, ve, z, yaw = ctl.command(t, n, e)
            client.moveByVelocityZAsync(vn, ve, z, dt * 4, airsim.DrivetrainType.MaxDegreeOfFreedom,
                                        airsim.YawMode(False, math.degrees(yaw)))
            if step > 0:
                client.simContinueForTime(dt)
            st = client.getMultirotorState()
            p = st.kinematics_estimated.position
            q = st.kinematics_estimated.orientation
            n, e, d = p.x_val, p.y_val, p.z_val
            cur_yaw = route.quat_wxyz_to_yaw(q.w_val, q.x_val, q.y_val, q.z_val)
            if step % 5 == 0:
                try:
                    if client.simGetCollisionInfo().has_collided:
                        collisions += 1
                except Exception:  # noqa: BLE001
                    pass
            # the chase camera: behind the drone along its heading, above it, looking at it
            cn, ce, cd = n - CHASE_BACK_M * math.cos(cur_yaw), e - CHASE_BACK_M * math.sin(cur_yaw), d - CHASE_UP_M
            pitch = -math.atan2(CHASE_UP_M, CHASE_BACK_M)
            sample = {"i": step, "t": (st.timestamp - ts0) / 1e9, "t_nominal": t, "n": n, "e": e, "d": d, "yaw": cur_yaw,
                      "fpv": None, "chase": None}
            try:
                if skip_grabs:
                    raise RuntimeError("test run without pictures")
                client.simSetCameraPose("Chase", airsim.Pose(airsim.Vector3r(cn, ce, cd), airsim.to_quaternion(pitch, 0.0, cur_yaw)),
                                        external=True)
            except Exception as ex:  # noqa: BLE001
                if cam_fail == 0:
                    log("simSetCameraPose failed:", repr(ex)[:200])
            for cam in ("fpv", "chase"):
                if skip_grabs:
                    break
                try:
                    img = grab(cam)
                except Exception as ex:  # noqa: BLE001
                    img = None
                    if cam_fail < 3:
                        log("picture failed", cam, repr(ex)[:200])
                if img is None:
                    cam_fail += 1
                    continue
                name = f"{cam}_{step:05d}.jpg"
                Image.fromarray(img).save(frames / name, quality=92)
                sample[cam] = name
                pics[cam] += 1
            samples.append(sample)
            if step % 75 == 0:
                log(f"step {step}/{nsteps} t={t:.1f}s pos=({n:.1f},{e:.1f},{-d:.1f}m up) pictures {pics}")
        wall = time.perf_counter() - wall0
        sim_s = samples[-1]["t"] - samples[0]["t"] if len(samples) > 1 else 0.0
        res["timing"] = {"wall_s": round(wall, 2), "sim_s": round(sim_s, 3), "real_time_factor": round(sim_s / wall, 3) if wall > 0 else None,
                         "steps": len(samples), "pictures_taken": pics,
                         "pictures_per_wall_second": round((pics["fpv"] + pics["chase"]) / wall, 2) if wall > 0 else None,
                         "steps_per_wall_second": round(len(samples) / wall, 2) if wall > 0 else None,
                         "camera_failures": cam_fail, "collision_checks_with_hit": collisions}
        res["stage"] = "done"
    except Exception:  # noqa: BLE001
        res["traceback"] = traceback.format_exc()
        res["stage"] = "flight_error"
        log(res["traceback"])
        rc = 1
    finally:
        try:
            client.simPause(False)
            client.hoverAsync()
            client.armDisarm(False)
            client.enableApiControl(False)
        except Exception:  # noqa: BLE001
            pass
        track = {"fps": a.fps, "plan_xy": route.plan_xy(res.get("start", {}).get("n", 0.0), res.get("start", {}).get("e", 0.0)),
                 "start_xy": [res.get("start", {}).get("n", 0.0), res.get("start", {}).get("e", 0.0)], "samples": samples}
        (out / "track.json").write_text(json.dumps(track))
        save_res()
    return rc


if __name__ == "__main__":
    sys.exit(main())''',
    "projectairsim_flight.py": r'''"""Project AirSim flight program (runs in the Python 3.10 environment that has the `projectairsim` package).

prepare_configs() (called by the notebook, needs no packages) copies the shipped scene and robot files and adds a front camera
("FrontCamera", on the nose, 640 x 480) and sets the clock so that the simulation runs slower than real time (RTF = `rtf`).

main(): connects, loads the scene, checks that a picture arrives, flies the route of route.py and saves pictures and track.json.
The clock runs on its own (slowed down); each pass of the loop takes the pictures, then reads the time and the pose, then sends
the next velocity command. The film is later made at a steady rate from the saved pictures by their simulated time (unreal_films.py).

Exit codes: 0 = flight done, 3 = no picture within the wait, 4 = no connection, 1 = error during the flight.
"""

import argparse
import asyncio
import json
import math
import re
import sys
import time
import traceback
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import route  # noqa: E402

SCENE_FILMS = "scene_films.jsonc"
SCENE_SHIPPED = "scene_basic_drone.jsonc"
ROBOT_FILMS = "robot_quadrotor_fastphysics_films.jsonc"
ROBOT_SHIPPED = "robot_quadrotor_fastphysics.jsonc"


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def strip_jsonc(text: str) -> str:
    """Remove // and /* */ comments outside of strings, and trailing commas."""
    out, i, n, in_str = [], 0, len(text), False
    while i < n:
        c = text[i]
        if in_str:
            out.append(c)
            if c == "\\":
                out.append(text[i + 1])
                i += 1
            elif c == '"':
                in_str = False
        elif c == '"':
            in_str = True
            out.append(c)
        elif text.startswith("//", i):
            while i < n and text[i] != "\n":
                i += 1
            continue
        elif text.startswith("/*", i):
            j = text.find("*/", i + 2)
            i = n if j < 0 else j + 1
        else:
            out.append(c)
        i += 1
    return re.sub(r",(\s*[}\]])", r"\1", "".join(out))


def prepare_configs(sim_config: Path, rtf: float) -> dict:
    """Write scene_films.jsonc and robot_quadrotor_fastphysics_films.jsonc next to the shipped files. Returns facts."""
    sim_config = Path(sim_config)
    robot = json.loads(strip_jsonc((sim_config / ROBOT_SHIPPED).read_text()))
    sensors = robot["sensors"]
    chase = next(s for s in sensors if s.get("id") == "Chase")
    facts = {"chase_size_before": [chase["capture-settings"][0].get("width"), chase["capture-settings"][0].get("height")]}
    chase["capture-settings"][0]["width"] = 640
    chase["capture-settings"][0]["height"] = 480
    front = json.loads(json.dumps(chase))
    front["id"] = "FrontCamera"
    front["origin"] = {"xyz": "0.3 0.0 0.0", "rpy-deg": "0 0 0"}
    front.pop("gimbal", None)  # fixed to the body: looks along the nose
    sensors.append(front)
    (sim_config / ROBOT_FILMS).write_text(json.dumps(robot, indent=2))
    scene = json.loads(strip_jsonc((sim_config / SCENE_SHIPPED).read_text()))
    for actor in scene["actors"]:
        if actor.get("robot-config") == ROBOT_SHIPPED:
            actor["robot-config"] = ROBOT_FILMS
    step_ns = int(scene["clock"]["step-ns"])
    scene["clock"]["real-time-update-rate"] = int(round(step_ns / rtf))
    scene["clock"]["pause-on-start"] = False
    (sim_config / SCENE_FILMS).write_text(json.dumps(scene, indent=2))
    facts.update({"step_ns": step_ns, "real_time_update_rate_ns": scene["clock"]["real-time-update-rate"], "rtf_wanted": rtf,
                  "drone_start": next((a.get("origin") for a in scene["actors"]), None)})
    return facts


def _get(obj, *names):
    """Value from a dict or an attribute object by the first name that exists."""
    for nme in names:
        if isinstance(obj, dict) and nme in obj:
            return obj[nme]
        if hasattr(obj, nme):
            return getattr(obj, nme)
    raise KeyError(names)


def pose_nedyaw(pose):
    t = _get(pose, "translation", "position")
    q = _get(pose, "rotation", "orientation")
    n, e, d = (float(_get(t, k)) for k in ("x", "y", "z"))
    w, x, y, z = (float(_get(q, k)) for k in ("w", "x", "y", "z"))
    return n, e, d, route.quat_wxyz_to_yaw(w, x, y, z)


async def amain(a) -> int:
    import numpy as np
    from PIL import Image
    from projectairsim import Drone, ProjectAirSimClient, World
    from projectairsim.types import ImageType
    from projectairsim.utils import unpack_image
    from projectairsim.drone import YawControlMode

    out = Path(a.out)
    frames = out / "frames"
    frames.mkdir(parents=True, exist_ok=True)
    res = {"stage": "start", "test": a.test, "traceback": None}

    def save_res():
        (out / "flight.json").write_text(json.dumps(res, indent=2, default=str))

    client = ProjectAirSimClient()
    t0 = time.time()
    while True:
        try:
            client.connect()
            break
        except Exception as e:  # noqa: BLE001
            if time.time() - t0 > a.connect_wait:
                res["stage"] = "no_connection"
                res["error"] = repr(e)
                save_res()
                return 4
            time.sleep(3)
    res["connected_after_s"] = round(time.time() - t0, 1)
    log("connected after", res["connected_after_s"], "s")

    def grab(drone, sensor):
        images = drone.get_images(sensor, [ImageType.SCENE])
        img = unpack_image(list(images.values())[0])
        arr = np.asarray(img)
        if arr.ndim == 3 and arr.shape[2] >= 3:
            arr = arr[:, :, :3][:, :, ::-1]  # unpack_image gives BGR; this is RGB
        return np.ascontiguousarray(arr)

    rc = 0
    samples = []
    try:
        world = drone = None
        used_scene = None
        for scene in (SCENE_FILMS, SCENE_SHIPPED):
            try:
                world = World(client, scene, delay_after_load_sec=2, sim_config_path=a.sim_config)
                drone = Drone(client, world, "Drone1")
                used_scene = scene
                break
            except Exception as e:  # noqa: BLE001
                log("scene", scene, "did not load:", repr(e)[:300])
                res.setdefault("scene_errors", []).append(f"{scene}: {e!r}"[:400])
        if drone is None:
            res["stage"] = "no_scene"
            save_res()
            return 1
        res["scene_used"] = used_scene
        res["clock_type"] = world.get_sim_clock_type()
        drone.enable_api_control()
        drone.arm()

        # ---- picture probe: chase, then the front camera; the front camera falls back to DownCamera ----
        res["stage"] = "probe"
        got, errors = {}, []
        front_names = ["FrontCamera", "DownCamera"] if used_scene == SCENE_FILMS else ["DownCamera"]
        t1 = time.time()
        while time.time() - t1 < a.picture_wait:
            for sensor in ["Chase"] + front_names:
                if sensor in got:
                    continue
                try:
                    got[sensor] = grab(drone, sensor)
                except Exception as e:  # noqa: BLE001
                    errors.append(f"{sensor}: {e!r}"[-300:])
            if "Chase" in got and any(f in got for f in front_names):
                break
            if "Chase" in got and time.time() - t1 > 20 and got:
                break  # a front camera that never answers: go on with what works
            time.sleep(2)
        front = next((f for f in front_names if f in got), None)
        res["probe"] = {"got": sorted(got), "front_sensor": front, "seconds": round(time.time() - t1, 1), "last_errors": errors[-3:]}
        if front is not None:
            arr = got[front].astype(float)
            top = arr[: max(1, arr.shape[0] // 5)]
            res["probe"]["front_top_rows_mean_rgb"] = [round(float(x), 1) for x in top.reshape(-1, 3).mean(axis=0)]
            res["probe"]["front_std"] = round(float(arr.std()), 1)
        log("probe:", res["probe"])
        if not got and not a.test:
            res["stage"] = "no_picture"
            save_res()
            return 3
        skip_grabs = a.test and not got
        res["skip_grabs"] = skip_grabs
        res["front_sensor_used"] = front
        res["front_note"] = ("front-facing camera on the nose" if front == "FrontCamera" else
                             "DownCamera (looks down) because no front camera picture arrived" if front == "DownCamera" else "no front picture")

        # ---- flight ----
        res["stage"] = "flight"
        save_res()
        cam_for = {"fpv": front, "chase": "Chase" if "Chase" in got or not skip_grabs else None}
        take = await drone.takeoff_async()
        sim0 = world.get_sim_time()
        wall0 = time.perf_counter()
        tstart = None  # sim time (s since sim0) when the route starts
        ctl = None
        pics = {"fpv": 0, "chase": 0}
        cam_fail = 0
        last_log = 0.0
        step = 0
        max_wall = a.max_wall
        n = e = d = yaw = 0.0
        cmd_task = None
        while True:
            if time.perf_counter() - wall0 > max_wall:
                res["stop_reason"] = "wall time limit"
                break
            sample = {"i": step, "fpv": None, "chase": None}
            if not skip_grabs:
                for cam in ("fpv", "chase"):
                    sensor = cam_for[cam]
                    if sensor is None:
                        continue
                    try:
                        img = grab(drone, sensor)
                    except Exception as ex:  # noqa: BLE001
                        cam_fail += 1
                        if cam_fail <= 3:
                            log("picture failed", sensor, repr(ex)[-200:])
                        continue
                    name = f"{cam}_{step:05d}.jpg"
                    Image.fromarray(img).save(frames / name, quality=92)
                    sample[cam] = name
                    pics[cam] += 1
            t_sim = (world.get_sim_time() - sim0) / 1e9
            n, e, d, yaw = pose_nedyaw(drone.get_ground_truth_pose())
            sample.update({"t": t_sim, "n": n, "e": e, "d": d, "yaw": yaw})
            if tstart is None and take.done():
                tstart = t_sim
                ctl = route.Controller(n, e, d, yaw)
                res["start"] = {"n": n, "e": e, "d": d, "yaw": yaw, "route_starts_at_sim_s": t_sim}
                # the takeoff already put the drone about 3 m up; the route's own climb is skipped
            samples.append(sample)
            if tstart is not None:
                tr = t_sim - tstart + route.CLIMB_S  # the route clock: skip the climb phase, start at the loop
                if tr > route.TOTAL_S:
                    res["stop_reason"] = "route finished"
                    break
                vn, ve, z, ywc = ctl.command(tr, n, e)
                z_target = res["start"]["d"]  # hold the height where the takeoff ended
                cmd_task = await drone.move_by_velocity_z_async(vn, ve, z_target, 0.5, YawControlMode.MaxDegreeOfFreedom, False, ywc)
            elif t_sim > 40.0:
                res["stop_reason"] = "takeoff did not finish within 40 simulated seconds"
                break
            step += 1
            await asyncio.sleep(0.01)
            if time.perf_counter() - last_log > 20:
                last_log = time.perf_counter()
                log(f"sim t={t_sim:.1f}s wall={last_log - wall0:.0f}s pos=({n:.1f},{e:.1f},{-d:.1f} up) pictures {pics} failures {cam_fail}")
        wall = time.perf_counter() - wall0
        sim_s = samples[-1]["t"] - samples[0]["t"] if len(samples) > 1 else 0.0
        res["timing"] = {"wall_s": round(wall, 2), "sim_s": round(sim_s, 3), "real_time_factor": round(sim_s / wall, 3) if wall > 0 else None,
                         "loop_passes": len(samples), "pictures_taken": pics,
                         "pictures_per_wall_second": round((pics["fpv"] + pics["chase"]) / wall, 2) if wall > 0 else None,
                         "camera_failures": cam_fail}
        res["stage"] = "done"
    except Exception:  # noqa: BLE001
        res["traceback"] = traceback.format_exc()
        res["stage"] = "flight_error"
        log(res["traceback"])
        rc = 1
    finally:
        try:
            drone.disarm()
            drone.disable_api_control()
        except Exception:  # noqa: BLE001
            pass
        try:
            client.disconnect()
        except Exception:  # noqa: BLE001
            pass
        st = res.get("start", {})
        track = {"fps": a.fps, "plan_xy": route.plan_xy(st.get("n", 0.0), st.get("e", 0.0)),
                 "start_xy": [st.get("n", 0.0), st.get("e", 0.0)], "samples": samples}
        (out / "track.json").write_text(json.dumps(track))
        save_res()
    return rc


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    ap.add_argument("--sim-config", required=True)
    ap.add_argument("--fps", type=int, default=15)
    ap.add_argument("--connect-wait", type=float, default=300.0)
    ap.add_argument("--picture-wait", type=float, default=90.0)
    ap.add_argument("--max-wall", type=float, default=900.0)
    ap.add_argument("--test", action="store_true", help="no pictures expected: fly the route anyway")
    a = ap.parse_args()
    return asyncio.run(amain(a))


if __name__ == "__main__":
    sys.exit(main())''',
}
for _name, _src in HELPERS.items():
    (WORK / "helpers" / _name).write_text(_src)
sys.path.insert(0, str(WORK / "helpers"))
say("helper programs written to", WORK / "helpers")

## Common tools for both simulators (starting a simulator, trying graphics modes, making the films)

In [ ]:
def ensure_sim_user():
    if not IS_ROOT:
        return Path.home()
    if sh(f"id {SIM_USER}") != 0:
        sh(f"useradd -m -s /bin/bash {SIM_USER}")
    return Path(f"/home/{SIM_USER}")


def make_venv(name, py, packages_cmds, log):
    """A separate Python environment made with uv, so old simulator clients do not clash with Colab's newer Python."""
    venv = WORK / "venvs" / name
    if (venv / "bin" / "python").exists():
        return venv
    venv.parent.mkdir(parents=True, exist_ok=True)
    sh(f"{sys.executable} -m pip install -q uv", log)
    rc = sh(f"{sys.executable} -m uv venv --seed --python {py} {venv}", log, timeout=900)
    if rc != 0:
        raise RuntimeError(f"could not make the Python {py} environment (see {log})")
    for c in packages_cmds:
        rc = sh(c.replace("PIP", f"{venv}/bin/pip"), log, timeout=1800)
        if rc != 0:
            raise RuntimeError(f"pip step failed: {c} (see {log})")
    return venv


def download_and_unpack(url, dest, log, zip_name):
    if dest.exists() and any(dest.iterdir()):
        return
    dest.mkdir(parents=True, exist_ok=True)
    z = WORK / "downloads" / zip_name
    z.parent.mkdir(parents=True, exist_ok=True)
    t = time.time()
    if sh(f"curl -fL --retry 5 -sS -o {z} {url}", log, timeout=1800) != 0:
        raise RuntimeError(f"download failed: {url}")
    say(f"downloaded {zip_name}: {z.stat().st_size / 1e9:.2f} GB in {time.time() - t:.0f} s")
    if sh(f"unzip -q -o {z} -d {dest}", log, timeout=1800) != 0:
        raise RuntimeError(f"unzip failed: {z}")
    z.unlink()
    if IS_ROOT:
        sh(f"chown -R {SIM_USER}:{SIM_USER} {dest}", log)
    sh(f"find {dest} -type f -path '*Binaries/Linux/*' -exec chmod +x {{}} + ; find {dest} -maxdepth 3 -name '*.sh' -exec chmod +x {{}} +", log)


def find_start_script(dest):
    r = subprocess.run(["bash", "-c", f"find {dest} -maxdepth 3 -name '*.sh' | head -n 1"], capture_output=True, text=True)
    p = r.stdout.strip()
    if not p:
        raise RuntimeError(f"no start script (.sh) found in {dest}")
    return p


VARIANTS = {
    "vulkan": "-RenderOffScreen -ResX=640 -ResY=480 -nosound -windowed",
    "opengl4": "-RenderOffScreen -opengl4 -ResX=640 -ResY=480 -nosound -windowed",
    "opengl": "-RenderOffScreen -opengl -ResX=640 -ResY=480 -nosound -windowed",
    "nullrhi": "-nullrhi -nosound",
}
_xvfb = {"p": None}


def ensure_xvfb():
    if _xvfb["p"] is None or _xvfb["p"].poll() is not None:
        _xvfb["p"] = subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1280x1024x24", "+extension", "GLX", "+render", "-noreset"],
                                      stdout=open(LOCAL_RESULTS / "xvfb.log", "ab"), stderr=subprocess.STDOUT, start_new_session=True)
        time.sleep(3)


def start_sim(script, args, sim_home, log):
    env_pairs = {"HOME": str(sim_home), "DISPLAY": ":99", "SDL_AUDIODRIVER": "dummy"}
    icd = S["vulkan"].get("nvidia_icd_file")
    if icd and S["vulkan"].get("nvidia_vulkan_device_found"):
        env_pairs["VK_ICD_FILENAMES"] = icd   # only the graphics card, never the software fallback
    inner = f"ulimit -c 0; exec bash {script} {args}"
    if IS_ROOT:
        cmd = ["runuser", "-u", SIM_USER, "--", "env"] + [f"{k}={v}" for k, v in env_pairs.items()] + ["bash", "-c", inner]
        env = None
    else:
        cmd = ["bash", "-c", inner]
        env = {**os.environ, **env_pairs}
    f = open(log, "ab")
    return subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, env=env, start_new_session=True, cwd=str(Path(script).parent))


def copy_unreal_logs(root, since, res_dir, prefix):
    """Unreal writes its own .log files inside the simulator folder; keep the ones written during this attempt."""
    n = 0
    for p in Path(root).rglob("*.log"):
        try:
            if p.stat().st_mtime >= since - 1:
                shutil.copyfile(p, res_dir / f"unreal_{prefix}__{p.name}")
                n += 1
        except Exception:  # noqa: BLE001
            pass
    return n


def try_variants(kind, script, sim_root, res_dir, client_cmd_for, variants, prepare_variant):
    """Start the simulator in each graphics mode until the flight program gets pictures and flies. Returns the list of attempts."""
    attempts = []
    for v in variants:
        att = {"variant": v, "args": VARIANTS[v]}
        attempts.append(att)
        run_dir = WORK / "runs" / kind / v
        shutil.rmtree(run_dir, ignore_errors=True)
        run_dir.mkdir(parents=True, exist_ok=True)
        say(f"[{kind}] starting the simulator with: {VARIANTS[v]}")
        prepare_variant(v)
        t0 = time.time()
        sim = start_sim(script, VARIANTS[v], ensure_sim_user(), res_dir / f"sim_{v}.log")
        clog = open(res_dir / f"client_{v}.log", "ab")
        client = subprocess.Popen(client_cmd_for(run_dir), stdout=clog, stderr=subprocess.STDOUT, start_new_session=True)
        died_at = None
        last_note = 0
        while client.poll() is None:
            time.sleep(2)
            if sim.poll() is not None:
                died_at = died_at or time.time()
                if time.time() - died_at > 12:
                    att["sim_died"] = True
                    say(f"[{kind}] the simulator process ended (exit code {sim.returncode}); stopping the flight program")
                    kill_group(client)
                    break
            if time.time() - last_note > 60:
                last_note = time.time()
                say(f"[{kind}] {v}: running for {time.time() - t0:.0f} s ...")
                sync_to_drive()
        att["client_rc"] = client.poll()
        att["sim_exit_code"] = sim.poll()
        att["seconds"] = round(time.time() - t0, 1)
        clog.close()
        att["unreal_logs_copied"] = copy_unreal_logs(sim_root, t0, res_dir, v)
        kill_group(sim)
        fj = run_dir / "flight.json"
        if fj.exists():
            shutil.copyfile(fj, res_dir / f"flight_{v}.json")
            att["stage"] = json.loads(fj.read_text()).get("stage")
        att["run_dir"] = str(run_dir)
        sync_to_drive()
        say(f"[{kind}] {v}: flight program exit code {att['client_rc']}, stage {att.get('stage')}, {att['seconds']} s")
        if att["client_rc"] == 0 or (att["client_rc"] == 1 and att.get("stage") == "flight_error"):
            att["used"] = True
            break
        time.sleep(3)
    return attempts


def color_note(rgb):
    if not rgb:
        return "no colour check (no picture)"
    r, g, b = rgb
    if b > r + 5:
        return f"the top of the front picture is bluish (red {r:.0f}, blue {b:.0f}), so the colour order looks right"
    return f"the top of the front picture is NOT bluish (red {r:.0f}, blue {b:.0f}); if the sky looks orange in the films, red and blue are swapped"


def finish_sim(kind, title, res_dir, attempts, extra_facts, camera_notes):
    """Make the films, contact sheets, summary.md and result.json for one simulator. Never claims a film that was not written."""
    import unreal_films

    used = next((a for a in attempts if a.get("used")), None)
    last = used or (attempts[-1] if attempts else None)
    result = {"simulator": title, "attempts": attempts, "facts": extra_facts, "films": {}, "problems": [], "test_mode": TEST}
    flight, info, quality = {}, {}, {}
    if last and last.get("run_dir"):
        run_dir = Path(last["run_dir"])
        try:
            flight = json.loads((run_dir / "flight.json").read_text())
        except Exception:  # noqa: BLE001
            pass
        tj = run_dir / "track.json"
        if tj.exists() and flight.get("stage") in ("done", "flight_error"):
            track = json.loads(tj.read_text())
            info = unreal_films.make_films(res_dir, run_dir / "frames", track, res_dir / "ffmpeg.log")
            quality = unreal_films.film_quality(res_dir, info)
            shutil.copyfile(tj, res_dir / "track.json")
    result["flight"] = flight
    result["film_info"] = info
    result["film_quality"] = quality
    films = list(info.get("films", {}))
    good = bool(films) and len(films) == 3 and flight.get("stage") == "done" and all(
        v.get("pictures_decoded", 0) >= 50 for v in info["films"].values()) and all(
        all(x > 5 for x in q.get("stds", [0])) for q in quality.values())
    if TEST:
        verdict = "TEST RUN WITHOUT PICTURES: the flight route was flown; pictures and films are not expected here."
    elif good:
        verdict = "WORKED: all three films were made."
    elif films:
        verdict = "PARTLY WORKED: some films were made, but not all checks passed (see below)."
    else:
        verdict = "DID NOT WORK: no film was made."
    result["verdict"] = verdict
    t = flight.get("timing", {})
    lines = [f"# {title}: first camera films", "", f"**Result: {verdict}**", ""]
    lines += ["## In plain English", ""]
    if used:
        lines.append(f"- The simulator drew its pictures in graphics mode **{used['variant']}** (modes tried, in order: "
                     + ", ".join(f"{a['variant']} ({'worked' if a.get('used') else 'failed'})" for a in attempts) + ").")
    else:
        lines.append("- No graphics mode gave pictures. Modes tried: " + (", ".join(a["variant"] for a in attempts) or "none") + ".")
    lines.append(f"- The graphics card's Vulkan driver: {'found' if S['vulkan'].get('nvidia_vulkan_device_found') else 'NOT found (only the slow software driver)'}"
                 + (f" ({S['gpu']['name']}, driver {S['gpu']['driver']})." if S.get('gpu') else "."))
    for n in camera_notes:
        lines.append(f"- {n}")
    if flight.get("probe", {}).get("fpv_top_rows_mean_rgb") or flight.get("probe", {}).get("front_top_rows_mean_rgb"):
        rgb = flight["probe"].get("fpv_top_rows_mean_rgb") or flight["probe"].get("front_top_rows_mean_rgb")
        lines.append(f"- Colour check: {color_note(rgb)}.")
    if t:
        lines.append(f"- The flight lasted {t.get('sim_s')} simulated seconds and took {t.get('wall_s')} real seconds. "
                     f"Simulated seconds per real second (the real-time factor): **{t.get('real_time_factor')}**.")
        lines.append(f"- Pictures taken: {t.get('pictures_taken')}. Pictures per real second (both cameras together): **{t.get('pictures_per_wall_second')}**. "
                     f"Camera failures: {t.get('camera_failures')}.")
    if info:
        lines.append(f"- The films are {(info.get('film_seconds') or 0):.1f} s long at {info.get('fps')} pictures per second of simulated time. "
                     f"Different pictures used: front camera {info.get('fpv_different_pictures_in_film')} "
                     f"({(info.get('fpv_pictures_per_sim_second') or 0):.1f} per simulated second), "
                     f"follow camera {info.get('chase_different_pictures_in_film')} ({(info.get('chase_pictures_per_sim_second') or 0):.1f} per simulated second). "
                     "If a number is below 15, some pictures are shown twice.")
    if flight.get("start"):
        lines.append(f"- Route: take off to about 3 m, one figure eight (12 m long, 6 m wide) with the nose along the direction of flight, then hover. "
                     f"Collision checks that reported a hit: {t.get('collision_checks_with_hit', 'not measured')}.")
    lines += ["", "## Files in this folder", ""]
    for name, f in info.get("films", {}).items():
        lines.append(f"- `{name}`: {f['pictures_decoded']} pictures, {f['bytes'] / 1e6:.1f} MB; contact sheet `{f['file']}`")
    if not info.get("films"):
        lines.append("- No films were written.")
    lines.append("- `summary.md` (this file), `result.json`, `track.json` (position of the drone for every picture), `flight_<mode>.json`, "
                 "and all logs: `sim_<mode>.log` (what the simulator printed), `client_<mode>.log` (the flight program), `unreal_<mode>__*.log` (the Unreal Engine log).")
    if quality:
        lines += ["", "## Film checks (12 pictures of each film)", ""]
        for n, q in quality.items():
            lines.append(f"- `{n}`: picture contrast (standard deviation, each should be above 5) {q.get('stds')}; "
                         f"mean change between neighbouring samples {q.get('neighbour_mean_abs_diff')}")
    problems = list(info.get("problems", []))
    if flight.get("traceback"):
        problems.append("flight program error: " + flight["traceback"].strip().splitlines()[-1])
    if flight.get("probe", {}).get("last_errors"):
        problems.append("picture errors: " + " | ".join(flight["probe"]["last_errors"]))
    if flight.get("stop_reason"):
        lines.append(f"\nThe flight stopped because: {flight['stop_reason']}.")
    if problems:
        lines += ["", "## Problems", ""] + [f"- {p}" for p in problems]
    if not good:
        lines += ["", "## Last 60 lines of the logs (because this did not fully work)", ""]
        for a in attempts:
            for pat in (f"client_{a['variant']}.log", f"sim_{a['variant']}.log"):
                lines += [f"### {pat}", "```", tail(res_dir / pat, 60).rstrip(), "```", ""]
            for p in sorted(res_dir.glob(f"unreal_{a['variant']}__*.log"))[:1]:
                lines += [f"### {p.name}", "```", tail(p, 60).rstrip(), "```", ""]
        for p in ("setup.log",):
            if (res_dir / p).exists():
                lines += [f"### {p}", "```", tail(res_dir / p, 60).rstrip(), "```", ""]
    (res_dir / "summary.md").write_text("\n".join(lines) + "\n")
    (res_dir / "result.json").write_text(json.dumps(result, indent=2, default=str))
    S["sims"][kind] = {"verdict": verdict, "films": films, "folder": str(DRIVE_RESULTS / kind if DRIVE_RESULTS else res_dir), "variant": used["variant"] if used else None}
    sync_to_drive()
    say(f"[{kind}] {verdict}")


def fail_sim(kind, title, res_dir, err):
    """A step before the flight failed (download, install, ...): write the summary with the last 60 lines of the logs."""
    lines = [f"# {title}: first camera films", "", "**Result: DID NOT WORK: the set-up stopped before the simulator could fly.**", "",
             f"Reason: `{err.strip().splitlines()[-1] if err.strip() else 'unknown'}`", "", "## Full error", "```", err.strip(), "```", "",
             "## Last 60 lines of the logs", ""]
    for p in sorted(res_dir.glob("*.log"))[:8]:
        lines += [f"### {p.name}", "```", tail(p, 60).rstrip(), "```", ""]
    (res_dir / "summary.md").write_text("\n".join(lines) + "\n")
    (res_dir / "result.json").write_text(json.dumps({"simulator": title, "verdict": "DID NOT WORK", "error": err, "test_mode": TEST}, indent=2))
    S["sims"][kind] = {"verdict": "DID NOT WORK (set-up)", "films": [], "folder": str(DRIVE_RESULTS / kind if DRIVE_RESULTS else res_dir), "variant": None}
    sync_to_drive()
    say(f"[{kind}] did not work: {err.strip().splitlines()[-1] if err.strip() else ''}")


def variants_to_try():
    return ["nullrhi"] if TEST else ["vulkan", "opengl4", "opengl"]

## Step 4. AirSim: install, start, fly, record
Downloads AirSim 1.8.1 (the "Blocks" world, Unreal Engine 4.27), installs its Python client in a Python 3.9 environment, writes `settings.json` (a camera on the drone's nose called `front`, and an external camera called `Chase` that is moved behind the drone at every step), then flies. The simulation is paused between steps, so the film time matches the simulated time even though drawing is slow.

In [ ]:
AIRSIM_RES = LOCAL_RESULTS / "airsim"
AIRSIM_STATE = {}
try:
    if (time.time() - T_START) / 60 > MAX_TOTAL_MINUTES:
        raise RuntimeError("skipped: the time limit for starting a new simulator was reached")
    say("=== AirSim ===")
    log = AIRSIM_RES / "setup.log"
    ensure_xvfb()
    home = ensure_sim_user()
    venv = make_venv("airsim", "3.9", [
        "PIP install -q --upgrade pip",
        "PIP install -q setuptools wheel msgpack-rpc-python numpy pillow opencv-python-headless",
        "PIP install -q --no-deps --no-build-isolation airsim==1.8.1",
        "PIP list",
    ], log)
    sh(f"{venv}/bin/python -c \"import airsim; print('airsim client', airsim.__version__)\"", log)
    sim_dir = WORK / "sims" / "airsim" / "Blocks"
    download_and_unpack(AIRSIM_URL, sim_dir, log, "airsim_blocks.zip")
    script = find_start_script(sim_dir)
    say("AirSim start script:", script)
    shot = {"CaptureSettings": [{"ImageType": 0, "Width": 640, "Height": 480, "FOV_Degrees": 90}]}
    settings = {
        "SettingsVersion": 1.2, "SimMode": "Multirotor", "ClockSpeed": 1,
        "CameraDefaults": shot,
        "Vehicles": {"Drone1": {"VehicleType": "SimpleFlight", "AutoCreate": True,
                                "Cameras": {"front": {**shot, "X": 0.3, "Y": 0.0, "Z": 0.0, "Pitch": 0.0, "Roll": 0.0, "Yaw": 0.0}}}},
        "ExternalCameras": {"Chase": {**shot, "X": -6.0, "Y": 0.0, "Z": -2.5, "Pitch": -22.0, "Roll": 0.0, "Yaw": 0.0}},
    }

    def prepare_airsim(variant):
        d = home / "Documents" / "AirSim"
        d.mkdir(parents=True, exist_ok=True)
        (d / "settings.json").write_text(json.dumps(settings, indent=2))
        if IS_ROOT:
            sh(f"chown -R {SIM_USER}:{SIM_USER} {home}")
        shutil.copyfile(d / "settings.json", AIRSIM_RES / "settings_used.json")

    def airsim_client_cmd(run_dir):
        cmd = [str(venv / "bin" / "python"), str(WORK / "helpers" / "airsim_flight.py"), "--out", str(run_dir), "--fps", str(FPS)]
        return cmd + (["--test", "--picture-wait", "15", "--connect-wait", "240"] if TEST else ["--picture-wait", "90", "--connect-wait", "240"])

    attempts = try_variants("airsim", script, sim_dir, AIRSIM_RES, airsim_client_cmd, variants_to_try(), prepare_airsim)
    AIRSIM_STATE["attempts"] = attempts
    finish_sim("airsim", "AirSim 1.8.1 (Blocks, Unreal Engine 4.27)", AIRSIM_RES, attempts,
               {"settings": settings},
               ["Cameras: `front` is fixed on the drone's nose and looks along it. `Chase` is an external camera that is moved at every step to "
                "6 m behind the drone (along its heading) and 2.5 m above it, looking at it (it follows the drone)."])
except Exception:  # noqa: BLE001
    err = traceback.format_exc()
    say("AirSim stopped:", err.strip().splitlines()[-1])
    fail_sim("airsim", "AirSim 1.8.1 (Blocks, Unreal Engine 4.27)", AIRSIM_RES, err)

## Step 5. Project AirSim: install, start, fly, record
Same idea with Project AirSim 1.0.1 (the "Blocks" world, Unreal Engine 5) and its own Python client (Python 3.10 environment). We add a `FrontCamera` on the drone's nose to a copy of the sample scene. The scene's own `Chase` camera follows the drone. The simulation clock runs ten times slower than real time, so that every picture can be drawn; the film time is the simulated time.

In [ ]:
PASIM_RES = LOCAL_RESULTS / "project_airsim"
PASIM_STATE = {}
try:
    if (time.time() - T_START) / 60 > MAX_TOTAL_MINUTES:
        raise RuntimeError("skipped: the time limit for starting a new simulator was reached")
    say("=== Project AirSim ===")
    log = PASIM_RES / "setup.log"
    ensure_xvfb()
    home = ensure_sim_user()
    venv = make_venv("pasim", "3.10", [
        "PIP install -q --upgrade pip",
        "PIP install -q projectairsim==1.0.1 numpy pillow",
        "PIP list",
    ], log)
    sh(f"{venv}/bin/python -c \"import projectairsim; print('projectairsim client ok')\"", log)
    src = WORK / "ProjectAirSim_src"
    if not src.exists():
        if sh(f"git clone --depth 1 --branch v1.0.1 https://github.com/iamaisim/ProjectAirSim.git {src}", log, timeout=900) != 0:
            raise RuntimeError("git clone of Project AirSim failed")
    sim_config = src / "client" / "python" / "example_user_scripts" / "sim_config"
    import projectairsim_flight

    facts = projectairsim_flight.prepare_configs(sim_config, RTF_PROJECT_AIRSIM)
    say("scene prepared:", facts)
    sim_dir = WORK / "sims" / "project_airsim" / "Blocks"
    download_and_unpack(PASIM_URL, sim_dir, log, "pasim_blocks.zip")
    script = find_start_script(sim_dir)
    say("Project AirSim start script:", script)

    def pasim_client_cmd(run_dir):
        cmd = [str(venv / "bin" / "python"), str(WORK / "helpers" / "projectairsim_flight.py"), "--out", str(run_dir), "--sim-config", str(sim_config),
               "--fps", str(FPS)]
        return cmd + (["--test", "--picture-wait", "15", "--connect-wait", "240", "--max-wall", "600"] if TEST else ["--picture-wait", "90", "--connect-wait", "300"])

    attempts = try_variants("project_airsim", script, sim_dir, PASIM_RES, pasim_client_cmd, variants_to_try(), lambda v: None)
    PASIM_STATE["attempts"] = attempts
    flight = {}
    try:
        used = next((a for a in attempts if a.get("used")), attempts[-1])
        flight = json.loads((Path(used["run_dir"]) / "flight.json").read_text())
    except Exception:  # noqa: BLE001
        pass
    notes = ["Cameras: " + (flight.get("front_note") or "front camera: not known (no flight data)") + ". The follow camera is the scene's own `Chase` camera "
             "(10 m behind the drone, 1 m above, it turns with the drone's heading)."]
    finish_sim("project_airsim", "Project AirSim 1.0.1 (Blocks, Unreal Engine 5)", PASIM_RES, attempts, facts, notes)
except Exception:  # noqa: BLE001
    err = traceback.format_exc()
    say("Project AirSim stopped:", err.strip().splitlines()[-1])
    fail_sim("project_airsim", "Project AirSim 1.0.1 (Blocks, Unreal Engine 5)", PASIM_RES, err)

## Step 6. Summary

In [ ]:
kill_group(_xvfb["p"])
minutes = (time.time() - T_START) / 60
top = ["# Colab run: AirSim and Project AirSim films", "", f"Run time: {minutes:.1f} minutes. Test mode: {TEST}.", ""]
print(f"\nThe run took {minutes:.1f} minutes.\n")
for kind in ("airsim", "project_airsim"):
    s = S["sims"].get(kind, {"verdict": "did not run", "films": [], "folder": "-"})
    print(f"{kind}:\n  result: {s['verdict']}\n  graphics mode that worked: {s.get('variant')}\n  folder: {s['folder']}")
    print("  films: " + (", ".join(s["films"]) if s["films"] else "none"))
    top += [f"## {kind}", f"- Result: {s['verdict']}", f"- Graphics mode that worked: {s.get('variant')}", f"- Folder: `{s['folder']}`",
            "- Films: " + (", ".join(s["films"]) if s["films"] else "none"), ""]
(LOCAL_RESULTS / "run_summary.md").write_text("\n".join(top) + "\n")
(LOCAL_RESULTS / "run.json").write_text(json.dumps(S, indent=2, default=str))
sync_to_drive()
print("\nAll results folder:", DRIVE_RESULTS or LOCAL_RESULTS)
print("Done")